# Gateway

The single door every tool request goes through. It checks the request, refuses any
day after the case's review day, hands the request to the right tool, and stamps the
reply with a fingerprint (`result_hash`) so a run can be replayed and checked later.

A bad request never crashes the run: it comes back as a failed reply (`ok=False`) with
the reason, which the model can read and correct. For case 7 the gateway also makes one
tool fail once, as the run settings say. The tools themselves never know about it.

Nothing here calls a model or a network; the gateway is an ordinary function call.

In [ ]:
#| default_exp gateway

In [ ]:
#| export
import hashlib
import json
import re
from collections.abc import Callable, Mapping
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

from pydantic import ValidationError

from agentic_forecast_review.generator import (
    Snapshot,
)
from agentic_forecast_review.schemas import (
    Case,
    EvidenceRowsArgs,
    FieldValue,
    RunSetting,
    ToolCall,
    ToolName,
    ToolResult,
)

## What a tool hands back

A tool returns named fields, the IDs of the rows behind them, and optionally a table.
It is called with the snapshot, the review day and its typed arguments, and raises
`ValueError` for a request it cannot answer (such as an unknown series).

In [ ]:
#| export
@dataclass(frozen=True)
class ToolOutput:
    fields: dict[str, FieldValue] = field(default_factory=dict)
    row_ids: tuple[str, ...] = ()
    table: tuple[dict[str, FieldValue], ...] = ()


Tool = Callable[[Snapshot, int, Any], ToolOutput]

## Call IDs and fingerprints

The gateway names each call: a short tool prefix and the call's number in the run, such
as `cmp2`. Claims cite these names in pointer tokens (`{ev:cmp2.n_actual}`).

The fingerprint is the SHA-256 of the reply's content in a fixed JSON form, so the same
answer always gets the same `result_hash`, whatever the call is named.

In [ ]:
#| export
CALL_PREFIX = {
    ToolName.RESOLVE_CONTEXT: "ctx",
    ToolName.COMPARE_RUNS: "cmp",
    ToolName.INSPECT_SERIES: "ins",
    ToolName.CHECK_QUALITY: "dq",
    ToolName.EVIDENCE_ROWS: "rows",
}


def _call_id(request: object, number: int) -> str:
    name = request.get("tool") if isinstance(request, Mapping) else None
    prefix = next((p for t, p in CALL_PREFIX.items() if t.value == name), "bad")
    return f"{prefix}{number}"


def result_hash(
    ok: bool,
    error: str | None,
    fields: Mapping[str, FieldValue],
    row_ids: tuple[str, ...],
    table: tuple[dict[str, FieldValue], ...],
) -> str:
    """SHA-256 of the reply's content in canonical JSON."""
    content = {
        "ok": ok,
        "error": error,
        "fields": dict(fields),
        "row_ids": list(row_ids),
        "table": list(table),
    }
    text = json.dumps(
        content, sort_keys=True, separators=(",", ":"), ensure_ascii=False
    )
    return hashlib.sha256(text.encode("utf-8")).hexdigest()

## The cutoff

Every day a request mentions: the ends of its day ranges, and the days inside row IDs
(`act__…__d0177` is day 177; `fc__…__d0168__h09` targets day 177). Any of them after the
review day is refused.

In [ ]:
#| export
_ROW_DAY = re.compile(r"^(act|fc)__.+__d(\d{4})(?:__h(\d{2}))?$")


def requested_days(args: Any) -> list[int]:
    """The days a request reaches: range ends and the days named by row IDs."""
    days = [
        getattr(args, name).end_day
        for name in ("period", "window", "days")
        if getattr(args, name, None) is not None
    ]
    if isinstance(args, EvidenceRowsArgs):
        for row_id in args.row_ids:
            match = _ROW_DAY.match(row_id)
            if match:
                kind, day, horizon = match.groups()
                days.append(int(day) + (int(horizon) if kind == "fc" else 0))
    return days

## The gateway

One gateway per run: it knows the case, its snapshot, the tools, and the case's run
setting. `call` takes a request exactly as the model sends it (plain JSON-like data)
and always returns a reply: the parsed call (when the request was valid) and its result.

In [ ]:
#| export
@dataclass(frozen=True)
class Reply:
    call: ToolCall | None
    result: ToolResult


class Gateway:
    def __init__(
        self,
        case: Case,
        snapshot: Snapshot,
        tools: Mapping[ToolName, Tool],
        setting: RunSetting | None = None,
    ) -> None:
        if snapshot.snapshot_id != case.snapshot_id:
            raise ValueError("snapshot does not match the case")
        self.case = case
        self.snapshot = snapshot
        self.tools = dict(tools)
        self.fail_once = setting.fail_first_call_to if setting else None
        self.calls = 0

    def call(self, request: object) -> Reply:
        """Check, run and fingerprint one tool request. Never raises for bad input."""
        self.calls += 1
        call_id = _call_id(request, self.calls)
        try:
            call = ToolCall.model_validate({"call_id": call_id, "args": request})
        except ValidationError as exc:
            return Reply(None, self._failed(call_id, _short(exc)))

        tool_name = call.args.tool
        late = [day for day in requested_days(call.args) if day > self.case.review_day]
        if late:
            error = f"day {max(late)} is after the review day {self.case.review_day}"
            return Reply(call, self._failed(call_id, error))
        if tool_name == self.fail_once:
            self.fail_once = None
            return Reply(call, self._failed(call_id, "timeout"))
        tool = self.tools.get(tool_name)
        if tool is None:
            return Reply(call, self._failed(call_id, f"{tool_name} is not available"))
        try:
            output = tool(self.snapshot, self.case.review_day, call.args)
        except ValueError as exc:
            return Reply(call, self._failed(call_id, str(exc)))

        digest = result_hash(True, None, output.fields, output.row_ids, output.table)
        result = ToolResult(
            call_id=call_id,
            ok=True,
            fields=output.fields,
            row_ids=output.row_ids,
            table=output.table,
            result_hash=digest,
        )
        return Reply(call, result)

    def _failed(self, call_id: str, error: str) -> ToolResult:
        digest = result_hash(False, error, {}, (), ())
        return ToolResult(call_id=call_id, ok=False, error=error, result_hash=digest)


def _short(exc: ValidationError) -> str:
    """Pydantic's errors, one line each, without links."""
    return "; ".join(
        f"{'.'.join(str(part) for part in error['loc'][1:]) or 'request'}: {error['msg']}"
        for error in exc.errors(include_url=False)
    )

Read the run settings file (one JSON line per case that has one).

In [ ]:
#| export
def load_run_settings(path: Path) -> dict[str, RunSetting]:
    """Run settings by case ID."""
    lines = path.read_text(encoding="utf-8").splitlines()
    settings = [RunSetting.model_validate_json(line) for line in lines if line]
    return {setting.case_id: setting for setting in settings}

## Examples

The real tools come in `07_tools`. Until then, two stand-in tools that return fixed
answers show what the gateway itself does.

In [ ]:
from agentic_forecast_review.cases import build_family


def stand_in_compare(snapshot: Snapshot, review_day: int, args: Any) -> ToolOutput:
    return ToolOutput(fields={"n_series": len(args.series)})


def stand_in_quality(snapshot: Snapshot, review_day: int, args: Any) -> ToolOutput:
    return ToolOutput(
        fields={"missing_days": 0},
        row_ids=("act__CA_2_FOODS__d0169",),
        table=({"day": 169, "rows": 1},),
    )


tools = {
    ToolName.COMPARE_RUNS: stand_in_compare,
    ToolName.CHECK_QUALITY: stand_in_quality,
}
family = build_family()
case = family.cases[0]
gateway = Gateway(case, family.snapshots[case.snapshot_id], tools)
compare = {
    "tool": "compare_runs",
    "series": ["CA_2_FOODS"],
    "metric": "mae",
    "period": {"start_day": 169, "end_day": 196},
    "window": {"start_day": 141, "end_day": 168},
}

A good request: the gateway names the call, runs the tool and stamps the reply.

In [ ]:
first = gateway.call(compare).result
print(first.call_id, first.ok, first.fields, first.result_hash[:12])
assert first.ok and first.call_id == "cmp1"

The same request again: a new call name, the same fingerprint.

In [ ]:
again = gateway.call(compare).result
print(again.call_id, again.result_hash[:12])
assert again.call_id == "cmp2" and again.result_hash == first.result_hash

A malformed request (no metric) comes back as a failed reply with the reason.

In [ ]:
no_metric = {key: value for key, value in compare.items() if key != "metric"}
bad = gateway.call(no_metric).result
print(bad.call_id, bad.ok, bad.error)
assert not bad.ok and "metric" in (bad.error or "")

Asking about a day after the review day (196) is refused.

In [ ]:
future = {**compare, "period": {"start_day": 169, "end_day": 203}}
late = gateway.call(future).result
print(late.call_id, late.ok, late.error)
assert not late.ok and "after the review day" in (late.error or "")

Case 7: its run setting makes `compare_runs` time out once; the retry works.

In [ ]:
settings = load_run_settings(Path("../cases/dev/run_settings.jsonl"))
case_7 = family.cases[6]
gateway_7 = Gateway(
    case_7, family.snapshots[case_7.snapshot_id], tools, settings[case_7.case_id]
)
replies = [gateway_7.call(compare).result for _ in range(2)]
for reply in replies:
    print(reply.call_id, reply.ok, reply.error)
assert [reply.ok for reply in replies] == [False, True]
assert replies[0].error == "timeout"